In [30]:
from pymilvus import connections

connections.connect(
    host="localhost",
    port="19530"
)

print("Connected to Milvus!")

Connected to Milvus!


In [31]:
from pymilvus import (
    connections, db,
    Collection, CollectionSchema, FieldSchema, DataType,
    utility
)

In [32]:
# ── 1. CONNECT (like opening a DB connection) ──────────────────────────
connections.connect(host="localhost", port="19530")

In [33]:
db.list_database()

['default', 'rag_db']

In [34]:
# ── 2. CREATE DATABASE (same as SQL: CREATE DATABASE rag_db) ───────────
if "rag_db" not in db.list_database():
    db.create_database("rag_db")

In [35]:
db.using_database("rag_db")

In [36]:
fields = [
    FieldSchema(
        name="id",
        dtype=DataType.INT64,
        is_primary=True,
        auto_id=True          # like AUTO_INCREMENT
    ),
    FieldSchema(
        name="text",
        dtype=DataType.VARCHAR,
        max_length=2000       # like VARCHAR(2000)
    ),
    FieldSchema(
        name="source",
        dtype=DataType.VARCHAR,
        max_length=500
    ),
    FieldSchema(
        name="category",
        dtype=DataType.VARCHAR,
        max_length=100
    ),
    FieldSchema(
        name="embedding",
        dtype=DataType.FLOAT_VECTOR,
        dim=384               # must match your embedding model's output size
    ),
]

In [37]:
schema = CollectionSchema(
    fields=fields,
    description="RAG document store"
)

In [38]:
# ── 4. CREATE COLLECTION (like CREATE TABLE) ───────────────────────────
COLLECTION_NAME = "documents"

In [39]:
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)   # drop if exists (like DROP TABLE)


In [40]:

# ── 5. CREATE INDEX on the vector column ──────────────────────────────
# In SQL you index a column to speed up lookups.
# In Milvus you MUST index the vector column — no index = no search.
#
# IVF_FLAT = good default (clusters vectors into buckets for fast search)
# COSINE   = similarity metric (best for text)

collection = Collection(name=COLLECTION_NAME, schema=schema)
print(f"Collection '{COLLECTION_NAME}' created")

collection.create_index(
    field_name="embedding",
    index_params={
        "index_type": "IVF_FLAT",
        "metric_type": "COSINE",
        "params": {"nlist": 128}
    }
)
print("Index created on embedding field")

Collection 'documents' created
Index created on embedding field


In [41]:
# insert into

In [42]:
# ! pip install -U sentence-transformers

In [43]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # outputs 384-dim vectors


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
# Your raw data — like rows in a table
rows = [
    {"text": "Milvus is an open-source vector database.",   "source": "milvus_docs", "category": "database"},
    {"text": "RAG stands for Retrieval Augmented Generation.", "source": "ai_glossary", "category": "ai"},
    {"text": "Embeddings are numerical representations of text.", "source": "ml_basics",  "category": "ml"},
]

In [45]:
# Generate the embedding for each row (fills the FLOAT_VECTOR column)
texts      = [r["text"]     for r in rows]
sources    = [r["source"]   for r in rows]
categories = [r["category"] for r in rows]
embeddings = embedder.encode(texts).tolist()

In [46]:
# INSERT INTO documents (text, source, category, embedding) VALUES (...)
collection.insert([texts, sources, categories, embeddings])
collection.flush()   # like COMMIT — persists data to disk
print(f"Inserted {len(rows)} rows")

Inserted 3 rows


In [47]:
# Search & Filter (like SELECT WHERE)

In [48]:
# Must load collection into memory before querying (like opening a table)
collection.load()


In [49]:
query = "What is a vector database?"
query_embedding = embedder.encode([query]).tolist()


In [50]:
query

'What is a vector database?'

In [51]:
# query_embedding

In [52]:
results = collection.search(
    data=query_embedding,
    anns_field="embedding",          # which column to search on
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,                         # TOP 3
    output_fields=["text", "source", "category"]  # like SELECT text, source, category
)

In [53]:
results

data: [[{'id': 464842922489807799, 'distance': 0.6754270792007446, 'entity': {'text': 'Milvus is an open-source vector database.', 'source': 'milvus_docs', 'category': 'database'}}, {'id': 464842922489807801, 'distance': 0.34119686484336853, 'entity': {'text': 'Embeddings are numerical representations of text.', 'source': 'ml_basics', 'category': 'ml'}}, {'id': 464842922489807800, 'distance': 0.1558305025100708, 'entity': {'text': 'RAG stands for Retrieval Augmented Generation.', 'source': 'ai_glossary', 'category': 'ai'}}]]

In [54]:
for hits in results:
    for hit in hits:
        print(f"Score: {hit.score:.3f} | {hit.entity.get('text')}")



Score: 0.675 | Milvus is an open-source vector database.
Score: 0.341 | Embeddings are numerical representations of text.
Score: 0.156 | RAG stands for Retrieval Augmented Generation.


In [55]:
# ── Filtered Search (vector search + WHERE clause) ────────────────────
# SQL analogy: SELECT * FROM documents WHERE category = 'ai'
#              ORDER BY similarity(embedding, ?) LIMIT 3


In [56]:
results_filtered = collection.search(
    data=query_embedding,
    anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    expr='category == "ai"',         # ← this is your WHERE clause
    output_fields=["text", "source", "category"]
)

In [57]:
for hits in results_filtered:
    for hit in hits:
        print(f"Score: {hit.score:.3f} | {hit.entity.get('text')}")



Score: 0.156 | RAG stands for Retrieval Augmented Generation.


In [58]:
# ── Scalar Query (no vector, just filter — like plain SELECT WHERE) ────
# SQL: SELECT * FROM documents WHERE source = 'milvus_docs'

scalar_results = collection.query(
    expr='source == "milvus_docs"',
    output_fields=["id", "text", "source"] #, "embedding"]
)
for row in scalar_results:
    print(row)

{'id': 464842922489807799, 'text': 'Milvus is an open-source vector database.', 'source': 'milvus_docs'}
